# Toy Brick Optimization Model (Small Example) - Snowflake Edition

This notebook demonstrates how to solve a toy brick set optimization problem using Gurobi on Snowflake data.

## Problem Statement
Given a collection of toy bricks (parts in specific colors), determine which sets can be built to maximize value while minimizing leftover pieces.

## Prerequisites
- Completed notebook 01_Prepare_Data
- Gurobi installed: `pip install gurobipy`
- Gurobi license (free trial available for small problems)

In [9]:
# Install required packages
!pip install gurobipy
!pip install snowflake-snowpark-python
!pip install pandas
!pip install numpy


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 7.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 7.2 MB/s eta 0:00:00a 0:00:01m
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [10]:
import snowflake.snowpark as snowpark
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, lit, sum as sum_, count
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import numpy as np

# Configuration
DATABASE = "TOY_BRICK_DB"
SCHEMA = "RAW_DATA"

# Create or get session
try:
    session = snowpark.Session.builder.getOrCreate()
except:
    connection_parameters = {
        "account": "your_account",
        "user": "your_user",
        "password": "your_password",
        "role": "your_role",
        "warehouse": "your_warehouse",
        "database": DATABASE,
        "schema": SCHEMA
    }
    session = Session.builder.configs(connection_parameters).create()

# Set the database and schema context
session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()

print(f"Connected to: {session.get_current_database()}.{session.get_current_schema()}")
print(f"Gurobi version: {gp.gurobi.version()}")

Connected to: "TOY_BRICK_DB"."RAW_DATA"
Gurobi version: (13, 0, 0)


## Step 1: Define Our Starting Inventory

For this small example, let's say we have 4 specific sets that we've disassembled into a bucket of bricks.

In [11]:
# Define the sets we own and have disassembled
owned_sets = [
    '60204-1',  # Police Station
    '60212-1',  # Barbecue Burn Out
    '60215-1',  # Fire Station
    '60221-1'   # Diving Yacht
]

print(f"Starting with {len(owned_sets)} owned sets:")
for set_num in owned_sets:
    result = session.sql(f"""
        SELECT set_name, year, theme_name, total_parts
        FROM v_set_parts
        WHERE set_num = '{set_num}'
        LIMIT 1
    """).collect()
    if result:
        print(f"  {set_num}: {result[0]['SET_NAME']} ({result[0]['TOTAL_PARTS']} parts)")

Starting with 4 owned sets:
  60204-1: City Hospital (861 parts)
  60212-1: Barbecue Burn Out (64 parts)
  60215-1: Fire Station (510 parts)
  60221-1: Diving Yacht (149 parts)


## Step 2: Calculate Available Inventory

Sum up all parts from our owned sets to get our bucket of bricks.

In [ ]:
# Create a temporary table with our inventory
owned_sets_str = "','".join(owned_sets)

session.sql("""
CREATE OR REPLACE TEMPORARY TABLE my_inventory AS
SELECT 
    part_num || '_' || color_id as part_color_id,
    part_num,
    color_id,
    part_name,
    color_name,
    SUM(quantity) as available_quantity
FROM v_set_parts
WHERE set_num IN ('{}')  
GROUP BY part_num, color_id, part_name, color_name
""".format(owned_sets_str)).collect()

# Get inventory summary
inventory_df = session.sql("""
    SELECT * FROM my_inventory ORDER BY available_quantity DESC LIMIT 20
""").to_pandas()

total_pieces = session.sql("SELECT SUM(available_quantity) as total FROM my_inventory").collect()[0]['TOTAL']
unique_part_colors = session.sql("SELECT COUNT(*) as cnt FROM my_inventory").collect()[0]['CNT']

print(f"Total pieces available: {total_pieces:,}")
print(f"Unique part-color combinations: {unique_part_colors:,}")
print("\nTop 20 most common pieces:")
print(inventory_df[['PART_NAME', 'COLOR_NAME', 'AVAILABLE_QUANTITY']].to_string(index=False))

SnowparkSQLException: (1304): 01c1b63a-0307-7651-0015-7a07125d079e: 000904 (42000): SQL compilation error: error line 3 at position 4
invalid identifier 'PART_COLOR_ID'
There are existing quoted column identifiers: ['"status"']. Please use one of them to reference the column. See more details on Snowflake identifier requirements https://docs.snowflake.com/en/sql-reference/identifiers-syntax

## Step 3: Define Candidate Sets

Select a subset of sets that could potentially be built (same theme, year range, etc.)

In [ ]:
# Get candidate sets from the same theme(s) and similar year range
# For small example, limit to 50 sets
candidate_sets_df = session.sql("""
SELECT DISTINCT
    s.set_num,
    s.name as set_name,
    s.year,
    s.theme_id,
    t.name as theme_name,
    s.num_parts
FROM sets s
JOIN themes t ON s.theme_id = t.id
WHERE s.year BETWEEN 2018 AND 2020
  AND s.theme_id IN (
      SELECT DISTINCT theme_id 
      FROM sets 
      WHERE set_num IN ('{}'))
  AND s.num_parts < 500  -- Keep it small for this example
ORDER BY s.num_parts DESC
LIMIT 50
""".format(owned_sets_str)).to_pandas()

print(f"Candidate sets to consider: {len(candidate_sets_df)}")
print("\nSample candidate sets:")
print(candidate_sets_df.head(10)[['SET_NUM', 'SET_NAME', 'YEAR', 'NUM_PARTS']].to_string(index=False))

## Step 4: Prepare Data for Optimization

Create the matrices needed for the optimization model.

In [ ]:
# Get requirements for candidate sets
candidate_set_list = candidate_sets_df['SET_NUM'].tolist()
candidate_sets_str = "','".join(candidate_set_list)

# Create requirements matrix
requirements_df = session.sql(f"""
SELECT 
    set_num,
    part_color_id,
    quantity as required_quantity
FROM set_part_requirements
WHERE set_num IN ('{candidate_sets_str}')
""").to_pandas()

# Get inventory
inventory_full_df = session.sql("SELECT * FROM my_inventory").to_pandas()

print(f"Requirements matrix: {len(requirements_df):,} rows")
print(f"Inventory: {len(inventory_full_df):,} unique part-color combinations")

## Step 5: Build the Optimization Model

### Decision Variables
- For each candidate set: binary variable (0 = don't build, 1 = build)

### Objective
- Maximize number of parts used (minimize leftover pieces)
- Or: Maximize number of sets built

### Constraints
- For each part-color combination: total used ≤ available quantity

In [ ]:
# Create optimization model
model = gp.Model("ToyBrickOptimization")

# Suppress Gurobi output for cleaner notebook
model.setParam('OutputFlag', 0)

# Create decision variables: one binary variable per candidate set
set_vars = {}
for set_num in candidate_set_list:
    set_vars[set_num] = model.addVar(vtype=GRB.BINARY, name=f"build_{set_num}")

print(f"Created {len(set_vars)} decision variables")

# Set objective: Maximize total parts used
# Alternative: Maximize number of sets built
objective_expr = gp.LinExpr()
for set_num in candidate_set_list:
    num_parts = candidate_sets_df[candidate_sets_df['SET_NUM'] == set_num]['NUM_PARTS'].values[0]
    objective_expr += num_parts * set_vars[set_num]

model.setObjective(objective_expr, GRB.MAXIMIZE)
print("Objective set: Maximize parts used")

# Add constraints: For each part-color, usage <= inventory
constraints_added = 0
for _, inv_row in inventory_full_df.iterrows():
    part_color_id = inv_row['PART_COLOR_ID']
    available = inv_row['AVAILABLE_QUANTITY']
    
    # Get all sets that need this part-color
    needs = requirements_df[requirements_df['PART_COLOR_ID'] == part_color_id]
    
    if len(needs) > 0:
        constraint_expr = gp.LinExpr()
        for _, req_row in needs.iterrows():
            set_num = req_row['SET_NUM']
            required_qty = req_row['REQUIRED_QUANTITY']
            constraint_expr += required_qty * set_vars[set_num]
        
        model.addConstr(constraint_expr <= available, name=f"inv_{part_color_id}")
        constraints_added += 1

print(f"Added {constraints_added} inventory constraints")

# Update model
model.update()
print("\nModel built successfully!")
print(f"Variables: {model.NumVars}")
print(f"Constraints: {model.NumConstrs}")

## Step 6: Solve the Optimization Problem

In [ ]:
# Solve
print("Solving optimization model...")
model.optimize()

# Check solution status
if model.status == GRB.OPTIMAL:
    print("\n" + "="*60)
    print("OPTIMAL SOLUTION FOUND!")
    print("="*60)
    
    # Get selected sets
    selected_sets = []
    for set_num, var in set_vars.items():
        if var.X > 0.5:  # Binary variable is 1
            selected_sets.append(set_num)
    
    print(f"\nSets to build: {len(selected_sets)}")
    print(f"Total parts used: {model.ObjVal:,.0f}")
    print(f"Parts available: {total_pieces:,}")
    print(f"Parts leftover: {total_pieces - model.ObjVal:,.0f}")
    print(f"Utilization: {(model.ObjVal / total_pieces * 100):.1f}%")
    
    print("\nSelected Sets:")
    for set_num in selected_sets:
        set_info = candidate_sets_df[candidate_sets_df['SET_NUM'] == set_num].iloc[0]
        print(f"  {set_num}: {set_info['SET_NAME']} ({set_info['NUM_PARTS']} parts)")
        
elif model.status == GRB.INFEASIBLE:
    print("Model is infeasible - cannot build any sets with available inventory")
else:
    print(f"Optimization ended with status {model.status}")

## Step 7: Analyze the Solution

Calculate detailed inventory usage and leftover pieces.

In [ ]:
if model.status == GRB.OPTIMAL and len(selected_sets) > 0:
    # Calculate parts used by part-color
    selected_sets_str = "','".join(selected_sets)
    
    usage_df = session.sql(f"""
    SELECT 
        r.part_color_id,
        i.part_num,
        i.part_name,
        i.color_name,
        SUM(r.quantity) as total_used,
        i.available_quantity,
        i.available_quantity - SUM(r.quantity) as leftover
    FROM set_part_requirements r
    JOIN my_inventory i ON r.part_color_id = i.part_color_id
    WHERE r.set_num IN ('{selected_sets_str}')
    GROUP BY r.part_color_id, i.part_num, i.part_name, i.color_name, i.available_quantity
    ORDER BY leftover DESC
    """).to_pandas()
    
    # Parts with biggest leftover
    print("\nTop 10 parts with most leftover pieces:")
    print(usage_df[usage_df['LEFTOVER'] > 0].head(10)[
        ['PART_NAME', 'COLOR_NAME', 'AVAILABLE_QUANTITY', 'TOTAL_USED', 'LEFTOVER']
    ].to_string(index=False))
    
    # Parts fully utilized
    fully_used = usage_df[usage_df['LEFTOVER'] == 0]
    print(f"\nPart-color combinations fully utilized: {len(fully_used)}")
    
    # Save solution to Snowflake
    solution_data = []
    for set_num in selected_sets:
        set_info = candidate_sets_df[candidate_sets_df['SET_NUM'] == set_num].iloc[0]
        solution_data.append({
            'SET_NUM': set_num,
            'SET_NAME': set_info['SET_NAME'],
            'YEAR': int(set_info['YEAR']),
            'NUM_PARTS': int(set_info['NUM_PARTS']),
            'SELECTED': 1
        })
    
    solution_df = pd.DataFrame(solution_data)
    session.create_dataframe(solution_df).write.mode('overwrite').save_as_table(
        'optimization_solution_small', 
        table_type='temporary'
    )
    
    print("\nSolution saved to temporary table: optimization_solution_small")

## Step 8: Alternative Objective - Maximize Number of Sets

In [ ]:
# Create a new model with different objective
model2 = gp.Model("ToyBrickOptimization_MaxSets")
model2.setParam('OutputFlag', 0)

# Same decision variables
set_vars2 = {}
for set_num in candidate_set_list:
    set_vars2[set_num] = model2.addVar(vtype=GRB.BINARY, name=f"build_{set_num}")

# Different objective: Maximize number of sets (not parts)
objective_expr2 = gp.quicksum(set_vars2.values())
model2.setObjective(objective_expr2, GRB.MAXIMIZE)

# Same constraints
for _, inv_row in inventory_full_df.iterrows():
    part_color_id = inv_row['PART_COLOR_ID']
    available = inv_row['AVAILABLE_QUANTITY']
    
    needs = requirements_df[requirements_df['PART_COLOR_ID'] == part_color_id]
    
    if len(needs) > 0:
        constraint_expr = gp.LinExpr()
        for _, req_row in needs.iterrows():
            set_num = req_row['SET_NUM']
            required_qty = req_row['REQUIRED_QUANTITY']
            constraint_expr += required_qty * set_vars2[set_num]
        
        model2.addConstr(constraint_expr <= available, name=f"inv_{part_color_id}")

model2.update()

# Solve
print("Solving with alternative objective (maximize set count)...")
model2.optimize()

if model2.status == GRB.OPTIMAL:
    print("\n" + "="*60)
    print("ALTERNATIVE SOLUTION (Maximize Set Count)")
    print("="*60)
    
    selected_sets2 = [set_num for set_num, var in set_vars2.items() if var.X > 0.5]
    
    total_parts_used2 = sum(
        candidate_sets_df[candidate_sets_df['SET_NUM'] == s]['NUM_PARTS'].values[0] 
        for s in selected_sets2
    )
    
    print(f"\nSets to build: {len(selected_sets2)}")
    print(f"Total parts used: {total_parts_used2:,.0f}")
    print(f"Parts leftover: {total_pieces - total_parts_used2:,.0f}")
    print(f"Utilization: {(total_parts_used2 / total_pieces * 100):.1f}%")
    
    print("\nSelected Sets:")
    for set_num in selected_sets2:
        set_info = candidate_sets_df[candidate_sets_df['SET_NUM'] == set_num].iloc[0]
        print(f"  {set_num}: {set_info['SET_NAME']} ({set_info['NUM_PARTS']} parts)")
    
    print("\n" + "="*60)
    print("Comparison:")
    print(f"Maximize Parts: {len(selected_sets)} sets, {model.ObjVal:,.0f} parts used")
    print(f"Maximize Sets:  {len(selected_sets2)} sets, {total_parts_used2:,.0f} parts used")
    print("="*60)

## Summary

This notebook demonstrated:
1. Loading toy brick data from Snowflake
2. Defining an inventory from owned sets
3. Building an optimization model with Gurobi
4. Solving to find which sets can be built
5. Comparing different objectives (maximize parts vs. maximize sets)

**Next Steps:**
- Proceed to notebook 03 for a larger-scale optimization problem
- Experiment with different owned set combinations
- Try different constraints (e.g., must use certain parts, prefer newer sets)